# GIS Transport MDA — Spatial Analysis
Jakarta Ride-Hailing Analytics: GIS Heatmaps, Demand Patterns, Fare Distributions

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import HeatMap
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

PG_HOST = os.getenv('PG_HOST', 'sources')
PG_PORT = os.getenv('PG_PORT', '5432')
PG_DB   = os.getenv('PG_DB',   'transport_db')
PG_USER = os.getenv('PG_USER', 'postgres')
PG_PASS = os.getenv('PG_PASSWORD', 'postgres')

engine = create_engine(f'postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}')
print('Connected to PostgreSQL')

## 1. Load enriched trip data

In [ ]:
df = pd.read_sql("""
    SELECT trip_id, origin_lat, origin_lng, dest_lat, dest_lng,
           origin_zone, dest_zone, vehicle_type, actual_fare,
           distance_km, surge_multiplier, weather,
           hour_of_day, day_of_week, processed_at
    FROM public.trips_enriched
    LIMIT 50000
""", engine)
print(f'Loaded {len(df):,} trips')
df.head()

## 2. Interactive pickup heatmap (Folium)

In [ ]:
m = folium.Map(location=[-6.2088, 106.8456], zoom_start=11, tiles='CartoDB positron')
heat_data = df[['origin_lat', 'origin_lng']].dropna().values.tolist()
HeatMap(heat_data, radius=10, blur=15, max_zoom=13).add_to(m)
folium.Marker([-6.2088, 106.8456], popup='Jakarta Center').add_to(m)
m

## 3. Fare distribution by vehicle type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby('vehicle_type')['actual_fare'].median().sort_values().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Median Fare by Vehicle Type'); axes[0].set_ylabel('IDR')
df.boxplot(column='actual_fare', by='vehicle_type', ax=axes[1])
axes[1].set_title('Fare Distribution'); plt.tight_layout(); plt.show()

## 4. Hourly demand heatmap

In [ ]:
pivot = df.groupby(['origin_zone', 'hour_of_day']).size().unstack(fill_value=0)
plt.figure(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, annot=False)
plt.title('Trip Volume by Zone and Hour of Day')
plt.tight_layout(); plt.show()

## 5. Predicted vs Actual Fare (model audit)

In [ ]:
preds = pd.read_sql("""
    SELECT tp.actual_fare, tp.predicted_fare, tp.origin_zone
    FROM public.trip_predictions tp
    WHERE tp.actual_fare IS NOT NULL
    LIMIT 5000
""", engine)
if len(preds) > 0:
    plt.figure(figsize=(7, 7))
    plt.scatter(preds['actual_fare'], preds['predicted_fare'], alpha=0.3, s=10)
    max_fare = max(preds['actual_fare'].max(), preds['predicted_fare'].max())
    plt.plot([0, max_fare], [0, max_fare], 'r--', label='Perfect prediction')
    plt.xlabel('Actual Fare (IDR)'); plt.ylabel('Predicted Fare (IDR)')
    plt.title('Predicted vs Actual Fare'); plt.legend(); plt.tight_layout(); plt.show()
else:
    print('No predictions yet. Run extract_predictions DAG first.')

## 6. Top Origin → Destination zone pairs

In [ ]:
od = df.groupby(['origin_zone', 'dest_zone']).size().reset_index(name='trips')
od = od.sort_values('trips', ascending=False).head(10)
od['route'] = od['origin_zone'] + ' → ' + od['dest_zone']
od.set_index('route')['trips'].plot(kind='barh', figsize=(10, 5), color='teal')
plt.title('Top 10 Origin → Destination Routes'); plt.xlabel('Trip Count')
plt.tight_layout(); plt.show()

## 7. Zone daily stats

In [ ]:
stats = pd.read_sql('SELECT * FROM public.zone_daily_stats ORDER BY stat_date DESC LIMIT 50', engine)
print(stats.to_string(index=False))